In [1]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.svm import SVC


import pandas as pd

from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ml.ModelOps import ModelOperations

raster_ops = RasterOperations()
model_ops = ModelOperations()

In [2]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train.csv")
y_test = pd.read_csv("csv_files/y_test.csv")

# compute indices
X_train = raster_ops.compute_ndvi_using_df(X_train, "NIR", "RED")
X_train = raster_ops.compute_ndbi(X_train, "NIR", "SWIR")
X_train = raster_ops.compute_rei(X_train, "NIR", "BLUE")
X_test = raster_ops.compute_ndvi_using_df(X_test, "NIR", "RED")
X_test = raster_ops.compute_ndbi(X_test, "NIR", "SWIR")
X_test = raster_ops.compute_rei(X_test, "NIR", "BLUE")


# create binary and discrete NDVI
X_train = raster_ops.create_ndvi_bin(X_train, "NDVI")
X_train = raster_ops.create_ndvi_discrete(X_train, "NDVI")
X_test = raster_ops.create_ndvi_bin(X_test, "NDVI")
X_test = raster_ops.create_ndvi_discrete(X_test, "NDVI")

In [3]:
X_train

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,NDVI_bin,NDVI_dis
0,373.5000,519.50000,389.0000,3034.0000,1991.0000,0.772714,-0.207562,0.002342,veg,high_veg
1,1917.0000,2370.00000,2264.0000,2617.0000,2415.0000,0.072321,-0.040143,0.000139,non_veg,low_veg
2,1616.0000,1715.83340,1704.0000,1832.0000,1769.2000,0.036199,-0.017439,0.000073,non_veg,low_veg
3,763.3333,1083.00000,919.6667,3341.0000,2946.3750,0.568299,-0.062765,0.001009,veg,high_veg
4,342.0000,508.33334,308.2000,2998.0000,1947.3334,0.813562,-0.212456,0.002583,veg,high_veg
...,...,...,...,...,...,...,...,...,...,...
1930,742.0000,1068.00000,967.5000,3268.5000,2751.2500,0.543201,-0.085925,0.001040,veg,high_veg
1931,1620.0000,1737.33340,1782.0000,1926.5000,2054.0000,0.038965,0.032031,0.000098,non_veg,low_veg
1932,1389.0000,1505.40000,1570.5000,1725.0000,1804.0000,0.046882,0.022386,0.000140,non_veg,low_veg
1933,1330.0000,1367.42860,1389.7500,1722.6666,2243.5000,0.106964,0.131319,0.000171,non_veg,low_veg


In [5]:
# Initialize the base models
best_params = {"C": 9.232675920286502, "max_iter": 160, "solver": "lbfgs"}
lr = LogisticRegression(**best_params)
gnb = GaussianNB()
svc = SVC(probability=True, random_state=42) 

pipeline_1 = model_ops.make_pipeline(lr)
pipeline_2 = model_ops.make_pipeline(gnb)
pipeline_3 = model_ops.make_pipeline(svc)

In [6]:
# Combine pipelines into a VotingClassifier
voting_clf = VotingClassifier(estimators=[
    ('lr', pipeline_1),
    ('gnb', pipeline_2),
    ('svc', pipeline_3)
], voting='soft')
voting_clf

VotingClassifier(estimators=[('lr',
                              Pipeline(steps=[('preprocessor',
                                               ColumnTransformer(transformers=[('num',
                                                                                Pipeline(steps=[('poly',
                                                                                                 PolynomialFeatures())]),
                                                                                ['BLUE',
                                                                                 'GREEN',
                                                                                 'RED',
                                                                                 'NIR',
                                                                                 'SWIR',
                                                                                 'NDVI',
                                                                                 'NDBI',
                                                                                 'REI']),
                                                                               ('onehot',
                                                                                OneHotEncoder(dtype=<class 'int'>),
                                                                                ['NDVI_bin']),
                                                                               ('ordinal',
                                                                                OrdinalEncoder(categories=[['low_veg',
                                                                                                            'medium_veg',
                                                                                                            'high_veg']],
                                                                                               dtype=...
                                                                                 'SWIR',
                                                                                 'NDVI',
                                                                                 'NDBI',
                                                                                 'REI']),
                                                                               ('onehot',
                                                                                OneHotEncoder(dtype=<class 'int'>),
                                                                                ['NDVI_bin']),
                                                                               ('ordinal',
                                                                                OrdinalEncoder(categories=[['low_veg',
                                                                                                            'medium_veg',
                                                                                                            'high_veg']],
                                                                                               dtype=<class 'int'>),
                                                                                ['NDVI_dis'])])),
                                              ('scale', MinMaxScaler()),
                                              ('dim_reduce',
                                               LinearDiscriminantAnalysis(n_components=3)),
                                              ('classifier',
                                               SVC(probability=True,
                                                   random_state=42))]))],
                 voting='soft')

In [7]:
for i, j in zip(["lr", "gnb", "svc"], range(3)):
    clf = voting_clf.estimators[j][1].fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)
    # calculate the accuracy
    print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)}")
    print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_test_pred)}")

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Train Accuracy: (0.937467700258398, 0.9378469707503807, 0.937467700258398, 0.9375970119680029)
Train Accuracy: (0.9256198347107438, 0.9265542124090291, 0.9256198347107438, 0.9258873031024559)


d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Train Accuracy: (0.9354005167958657, 0.9358813354086657, 0.9354005167958657, 0.9355260172950663)
Train Accuracy: (0.9338842975206612, 0.9340984680507962, 0.9338842975206612, 0.9339686954867891)


d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Train Accuracy: (0.9359173126614987, 0.9367937842691959, 0.9359173126614987, 0.9361516139063547)
Train Accuracy: (0.9256198347107438, 0.9270178151846417, 0.9256198347107438, 0.9259649986104941)


In [8]:
# Fit the VotingClassifier
voting_clf.fit(X_train, y_train)

# Predict the labels of the test set
y_train_pred = voting_clf.predict(X_train)
y_test_pred = voting_clf.predict(X_test)

# Calculate the accuracy of the VotingClassifier
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)}")
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_test_pred)}")

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\preprocessing\_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\preprocessing\_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


Train Accuracy: (0.937467700258398, 0.9379242656795944, 0.937467700258398, 0.937615754009898)
Train Accuracy: (0.9297520661157025, 0.9299764634739202, 0.9297520661157025, 0.9298417389547133)
